# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "11-01-2021"
end_date = "07-25-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

## Read Metadata 

In [3]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

# print(len(metadata)) 
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

10340
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')


In [4]:
# Get list of genotypes

# os.chdir(references)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

# genotypes = ["B3.13", "D1.1", "D1.3"]

genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1 (hard-coded)

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

## Get genotype

In [5]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
# display(metadata)

8406


## Get specific geolocation

In [6]:
os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")

# Double-check state with genbank_mapping
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="left")
# Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x 
                                                                else x)
metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
metadata["Geo_Location"] = metadata["name_state_genbank"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x.split(" ")[-1]
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                        else 
                                                        "USA"
                                                        )

# If USA-, delete -
metadata["Geo_Location"] = metadata["Geo_Location"].apply(lambda x: x.replace("-", "") if x == "USA-" else x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
# metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,seg,genbank_acc,genbank_seg,genbank_name,name_state_genbank,Geo_Location
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,"7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,HA,PP740722.1,4.0,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,"7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,SRR28752446_MP_cns.fa,Consensus_SRR28752446_MP_cns_threshold_0.5_qua...,MP,PP740723.1,7.0,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
2,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,"7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,SRR28752446_NA_cns.fa,Consensus_SRR28752446_NA_cns_threshold_0.5_qua...,NaN,PP740724.1,6.0,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
3,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,"7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,SRR28752446_NP_cns.fa,Consensus_SRR28752446_NP_cns_threshold_0.5_qua...,NP,PP740725.1,5.0,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
4,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024-03-16,...,"7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report,SRR28752446_NS_cns.fa,Consensus_SRR28752446_NS_cns_threshold_0.5_qua...,NS,PP740726.1,8.0,A/blackbird/Texas/24-008354-001/2024,Texas,USA-TX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48121,SRR34710291,WGS,148.60,103689833,PRJNA1207547,SAMN50195801,Viral,38900497,USDA-NVSL,2025,...,"6, 9, 0, 2, 6, 8, 5, 11",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
48122,SRR34710294,WGS,148.18,93176431,PRJNA1207547,SAMN50195796,Viral,35091439,USDA-NVSL,2025,...,"7, 9, 7, 5, 3, 9, 17, 5",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
48123,SRR34710295,WGS,148.27,100227039,PRJNA1207547,SAMN50195795,Viral,37319422,USDA-NVSL,2025,...,"4, 12, 11, 10, 7, 15, 11, 9",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
48124,SRR34710296,WGS,147.95,100116637,PRJNA1207547,SAMN50195784,Viral,37411408,USDA-NVSL,2025,...,"7, 7, 43, 6, 7, 2, 10, 7",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA


## Collection Dates

In [7]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x, default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

0        2024-03-16
1        2024-03-16
2        2024-03-16
3        2024-03-16
4        2024-03-16
            ...    
48121          2025
48122          2025
48123          2025
48124          2025
48125          2025
Name: Collection_Date, Length: 48126, dtype: object


## Get host type

In [8]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
           wild_avian domestic_avian               cattle        feline  \
0    great_horned_owl       pheasant            dairy_cow           cat   
1        common_raven         turkey               cattle  domestic_cat   
2       cooper's_hawk        chicken  cattle milk product     feral_cat   
3        coopers_hawk          goose          bovine_milk        feline   
4             peafowl    guinea_fowl              bovine   domestic-cat   
..                ...            ...                  ...           ...   
802          anas sp.            NaN                  NaN           NaN   
803         anser sp.            NaN                  NaN           NaN   
804        corvus sp.            NaN                  NaN           NaN   
805   phasianidae sp.            NaN                  NaN           NaN   
806      loon, common            NaN                  NaN           NaN   

      other_mammal       human         other  new  
0       deer mouse  washington         mixed

In [9]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

## Make names using all the attributes we collected

In [10]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: x if "-" not in x else str(dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(x, default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(x, default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

0        >SRR28752446|A/blackbird/United States/24-0083...
1        >SRR28752446|A/blackbird/United States/24-0083...
2        >SRR28752446|A/blackbird/United States/24-0083...
3        >SRR28752446|A/blackbird/United States/24-0083...
4        >SRR28752446|A/blackbird/United States/24-0083...
                               ...                        
48121    >SRR34710291|A/great_black-backed_gull/United ...
48122    >SRR34710294|A/loon,_common/United States/25-0...
48123    >SRR34710295|A/common_eider/United States/25-0...
48124    >SRR34710296|A/american_robin/United States/25...
48125    >SRR34709337|A/cattle/United States/25-019681-...
Name: Name, Length: 48126, dtype: object

In [11]:
# Drop duplicate runs 

metadata["Partials"] = metadata["isolate"].apply(partial_isolate)
metadata = metadata.drop_duplicates(subset=["Partials", "years"], keep="last") # Isolates may be identical

In [12]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

               Run Assay Type  AvgSpotLen      Bases    BioProject  \
7      SRR28752446        WGS      146.11   93605195  PRJNA1102327   
15     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
23     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
31     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
39     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
...            ...        ...         ...        ...           ...   
48121  SRR34710291        WGS      148.60  103689833  PRJNA1207547   
48122  SRR34710294        WGS      148.18   93176431  PRJNA1207547   
48123  SRR34710295        WGS      148.27  100227039  PRJNA1207547   
48124  SRR34710296        WGS      147.95  100116637  PRJNA1207547   
48125  SRR34709337        WGS      149.05   60591683  PRJNA1219588   

          BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
7      SAMN41019184          Viral  30074178   USDA-NVSL      2024-03-16  ... 

## Make FASTA files

In [13]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [14]:
# Create fasta files 

os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty
        output_path = complete_files + pair + "_" + date_range + "_andersen.fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR28834929|A/snow_goose/California/24-004881-004-original/2024|H5N1|USA-CA|2024-01-18|wild_avian|B3.2
>SRR28834940|A/snow_goose/California/24-004881-002-original/2024|H5N1|USA-CA|2024-01-18|wild_avian|B3.2
>SRR32006905|A/chicken/California/24-038987-008/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006907|A/chicken/California/24-038987-007/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006908|A/chicken/California/24-038987-006/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006909|A/chicken/California/24-038987-005/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006910|A/chicken/California/24-038987-004/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006911|A/chicken/California/24-038987-003/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006912|A/chicken/California/24-038987-002/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR32006913|A/chicken/California/24-038987-001/2024|H5N1|USA-CA|2024-12-26|domestic_avian|B3.2
>SRR31301027|A/chicken/P